# Universal AC-MOT V4 — Colab T4 run (frozen method, no tuning)
Runs ONLY: (1) Faster R-CNN detection caches for the locked unseen-detector / UAVDT / NMS-audit evaluations, (2) the official T4 latency benchmark.
The policy is the frozen V4 (tag `universal-acmot-v4-freeze`); the hash check below must print `True` for every file.
Does not touch `/content/Universal-ACMOT` (older experimental workspace).

In [ ]:
# 1. GPU + Drive
import subprocess, torch
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv'],capture_output=True,text=True).stdout)
assert torch.cuda.is_available() and 'T4' in torch.cuda.get_device_name(0), 'Official timing requires a T4 runtime'
from google.colab import drive; drive.mount('/content/drive')

In [ ]:
# 2. Fresh clone of the pushed branch + lock hash verification
%cd /content
!rm -rf /content/Universal-ACMOT-v4
!git clone -q --branch universal-adapters-v1 https://github.com/AhmedCode110/AC-MOT.git /content/Universal-ACMOT-v4
%cd /content/Universal-ACMOT-v4
!git log --oneline -1
import json, hashlib
L = json.load(open('research/TESTDEV_LOCK_V4.json'))
ok = {p: hashlib.sha256(open(p,'rb').read()).hexdigest() == h for p, h in L['file_sha256'].items()}
print(ok); assert all(ok.values()), 'LOCK VIOLATION — stop'

In [ ]:
# 3. Pinned packages (same versions as the Mac development env)
!pip -q install ultralytics==8.3.200 lap==0.5.12
import ultralytics, torchvision; print(ultralytics.__version__, torch.__version__, torchvision.__version__)

In [ ]:
# 4. Weights (+ sha256 vs the lock)
import os, shutil
os.makedirs('weights', exist_ok=True)
for w in ['yolov8n.pt', 'rtdetr-l.pt']:
    old = f'/content/Universal-ACMOT/{w}'
    if os.path.exists(old): shutil.copy(old, w)
!test -f yolov8n.pt || wget -q https://github.com/ultralytics/assets/releases/download/v8.3.0/yolov8n.pt
!test -f rtdetr-l.pt || wget -q https://github.com/ultralytics/assets/releases/download/v8.3.0/rtdetr-l.pt
!test -f weights/fasterrcnn_resnet50_fpn_v2_coco-dd69338a.pth || wget -q -P weights https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_v2_coco-dd69338a.pth
EXPECT = {'yolov8n.pt':'f59b3d833e2ff32e194b5bb8e08d211dc7c5bdf144b90d2c8412c47ccfc83b36',
          'rtdetr-l.pt':'6de60b10d4bc566f00cda0f5b4d64afe4b66d48dc9695d2171effb7859d8e73f',
          'weights/fasterrcnn_resnet50_fpn_v2_coco-dd69338a.pth':'dd69338a24b8d7381807e247652bdc356325bcbaf1cd3e092e00e0a1a58706bf'}
for p, h in EXPECT.items():
    print(p, hashlib.sha256(open(p,'rb').read()).hexdigest() == h)

In [ ]:
# 5. Datasets -> local disk (fast I/O). Paths on the a7medgouda1 Drive.
D = '/content/drive/MyDrive'
VAL_SRC = f'{D}/AC-MOT-shared/AC-MOT-data/VisDrone2019-MOT-val'
TD_SRC  = f'{D}/visdrone/VisDrone_Zips/VisDrone2019-MOT-test-dev/VisDrone2019-MOT-test-dev'
UA_SRC  = f'{D}/AC-MOT-shared/UAVDT_EXTERNAL_GENERALIZATION/data/extracted/UAV-benchmark-M/UAV-benchmark-M'
UA_TEST = ['M0203','M0205','M0208','M0209','M0403','M0601','M0602','M0606','M0701','M0801',
           'M0802','M1001','M1004','M1007','M1009','M1101','M1301','M1302','M1303','M1401']
!test -d /content/VisDrone2019-MOT-val || cp -r "$VAL_SRC" /content/VisDrone2019-MOT-val
!test -d /content/VisDrone2019-MOT-test-dev || cp -r "$TD_SRC" /content/VisDrone2019-MOT-test-dev
os.makedirs('/content/uavdt_view/sequences', exist_ok=True)
for s in UA_TEST:
    if not os.path.isdir(f'/content/uavdt_view/sequences/{s}'):
        shutil.copytree(f'{UA_SRC}/{s}', f'/content/uavdt_view/sequences/{s}')
for p in ['/content/VisDrone2019-MOT-val','/content/VisDrone2019-MOT-test-dev','/content/uavdt_view']:
    print(p, len(os.listdir(p + '/sequences')), 'sequences')

In [ ]:
# 6. Faster R-CNN caches on T4 (inference only; no metrics here)
OUT = f'{D}/Universal-ACMOT-Results/v4_t4/frcnn_cache'
W = 'weights/fasterrcnn_resnet50_fpn_v2_coco-dd69338a.pth'
jobs = [
  ('/content/VisDrone2019-MOT-val',      f'{OUT}/val_nms045/fasterrcnn',     '0.45', '640 736'),  # V4 request (locked transfer)
  ('/content/VisDrone2019-MOT-test-dev', f'{OUT}/testdev_nms045/fasterrcnn', '0.45', '640 736'),  # held-out transfer
  ('/content/uavdt_view',                f'{OUT}/uavdt_nms045/fasterrcnn',   '0.45', '736'),      # cross-dataset transfer
  ('/content/VisDrone2019-MOT-val',      f'{OUT}/val_nms050/fasterrcnn',     '0.5',  '736'),      # NMS audit: torchvision-native 0.5
]
for ds, out, nms, res in jobs:
    !PYTHONPATH=. python -u tools/cache_detections.py --weights $W --dataset "$ds" --output-dir "$out" --nms $nms --resolutions $res 2>&1 | grep -v -i warn

In [ ]:
# 7. OFFICIAL T4 benchmark of the frozen V4 pipeline (same commit as above)
!PYTHONPATH=. python tools/t4_benchmark.py --dataset /content/VisDrone2019-MOT-val --sequence uav0000137_00458_v \
   --weights yolov8n.pt rtdetr-l.pt weights/fasterrcnn_resnet50_fpn_v2_coco-dd69338a.pth \
   --levels 640 736 832 --trackers bytetrack botsort --frames 300 --warmup 20 \
   --out "$D/Universal-ACMOT-Results/v4_t4/t4_benchmark_v4.json" 2>&1 | grep -v -i warn
!git rev-parse HEAD > "$D/Universal-ACMOT-Results/v4_t4/COMMIT.txt"
!pip freeze | grep -i -E "torch|ultralytics|numpy|opencv|lap" > "$D/Universal-ACMOT-Results/v4_t4/PACKAGES.txt"
print(open(f'{D}/Universal-ACMOT-Results/v4_t4/COMMIT.txt').read())